In [ ]:
"""
08_explainability_and_fault_isolation.ipynb

Purpose:
Explain the final NetGuard model using SHAP and convert model explanations
into operator-friendly fault isolation outputs.

What this notebook does:
1. Load the final trained model
2. Load final training and test data
3. Generate SHAP explanations
4. Produce global explainability outputs
5. Produce local explanations for selected test cases
6. Convert top features into fault categories
7. Generate isolation summaries and recommended checks

Outputs saved to:
    ../reports/explainability/
    ../artifacts/explainability/
"""

# 1. Imports

# sys is used only to print Python version
import sys

# Path helps build clean file paths
from pathlib import Path

# warnings are hidden to keep notebook output cleaner
import warnings

# joblib loads the saved final model
import joblib

# numpy and pandas are used for data handling
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# SHAP is required for explainability
try:
    import shap
except Exception as exc:
    raise ImportError("SHAP is required for this notebook. Install with: pip install shap") from exc

# display() looks nicer in notebooks, but print() is used if unavailable
try:
    from IPython.display import display
except Exception:
    display = None

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("SHAP   :", shap.__version__)


# 2. Paths

PROJECT_ROOT = Path.cwd().parent
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"

# Final locked model from notebook 07
MODEL_PATH = PROJECT_ROOT / "artifacts" / "final_model" / "lightgbm_final_model.joblib"

# Final test predictions from notebook 07
PREDS_PATH = PROJECT_ROOT / "artifacts" / "final_predictions" / "final_test_predictions.csv"

REPORTS_PATH = PROJECT_ROOT / "reports" / "explainability"
ARTIFACTS_PATH = PROJECT_ROOT / "artifacts" / "explainability"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SPLITS_PATH  :", SPLITS_PATH)
print("MODEL_PATH   :", MODEL_PATH)
print("PREDS_PATH   :", PREDS_PATH)
print("REPORTS_PATH :", REPORTS_PATH)
print("ARTIFACTS    :", ARTIFACTS_PATH)


# 3. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


def normalize_shap_output(shap_values, class_labels):
    """
    Standardize SHAP output into:
        {class_label: 2D array [n_samples, n_features]}

    SHAP can return values in different formats depending on the model/library.
    This helper makes the output easier to use later.
    """
    # Case 1: SHAP returns a list, one array per class
    if isinstance(shap_values, list):
        return {
            cls: np.array(shap_values[i])
            for i, cls in enumerate(class_labels)
        }

    arr = np.array(shap_values)

    # Case 2: SHAP returns a 3D array
    if arr.ndim == 3:
        # Common format: (n_samples, n_features, n_classes)
        if arr.shape[2] == len(class_labels):
            return {
                cls: arr[:, :, i]
                for i, cls in enumerate(class_labels)
            }

        # Alternative format: (n_classes, n_samples, n_features)
        if arr.shape[0] == len(class_labels):
            return {
                cls: arr[i, :, :]
                for i, cls in enumerate(class_labels)
            }

    raise ValueError(
        f"Unexpected SHAP output shape/type: type={type(shap_values)}, shape={getattr(arr, 'shape', None)}"
    )


def get_top_features_for_row(shap_matrix, feature_names, row_index, top_n=10):
    """
    For one row, return the top features with the largest absolute SHAP values.

    These are the features that contributed most strongly to the prediction.
    """
    vals = shap_matrix[row_index]

    out = pd.DataFrame(
        {
            "feature": feature_names,
            "shap_value": vals,
            "abs_shap_value": np.abs(vals),
        }
    ).sort_values("abs_shap_value", ascending=False).head(top_n).reset_index(drop=True)

    return out


def detect_feature_group(feature_name: str) -> str:
    """
    Map a feature name to a broader feature group.

    This makes explanations easier for humans to understand.
    """
    if feature_name == "location_num":
        return "location"
    if feature_name.startswith("severity_type_"):
        return "severity_type"
    if feature_name.startswith("event_type_"):
        return "event_type"
    if feature_name.startswith("resource_type_"):
        return "resource_type"
    if feature_name.startswith("log_feature_") and feature_name.endswith("_vol"):
        return "log_feature_volume"
    if feature_name.startswith("log_feature_"):
        return "log_feature_presence"
    return "engineered"


def infer_fault_category(top_features_df: pd.DataFrame) -> str:
    """
    Convert top explanatory features into a simpler fault category.

    This is the step that turns model explanation into an operator-friendly
    isolation label.
    """
    groups = top_features_df["feature"].apply(detect_feature_group)
    counts = groups.value_counts()

    top_group = counts.index[0]

    if top_group == "location":
        return "Site/Location Hotspot"
    if top_group in ["log_feature_presence", "log_feature_volume"]:
        return "Software/Log Anomaly"
    if top_group == "resource_type":
        return "Resource/Capacity Stress"
    if top_group == "event_type":
        return "Alarm Burst / Incident Cascade"
    if top_group == "severity_type":
        return "Historical Severity Pattern"
    return "Mixed Operational Signals"


def build_isolation_summary(top_features_df: pd.DataFrame, predicted_class: int) -> str:
    """
    Build a short plain-English summary describing why the model predicted
    the given severity class.
    """
    category = infer_fault_category(top_features_df)
    top_list = top_features_df["feature"].head(3).tolist()
    top_text = ", ".join(top_list)

    return (
        f"The model predicted severity class {predicted_class} mainly due to signals related to "
        f"{category.lower()}. The strongest contributing features were: {top_text}."
    )


def build_recommended_checks(category: str):
    """
    Suggest practical checks for engineers based on the inferred fault category.
    """
    if category == "Site/Location Hotspot":
        return [
            "Inspect the affected site and nearby dependent nodes.",
            "Check whether the same location has repeated recent incidents.",
            "Verify site power, connectivity, and local configuration health.",
        ]

    if category == "Software/Log Anomaly":
        return [
            "Review recent log spikes and abnormal software events.",
            "Check whether the issue followed a software update or configuration change.",
            "Inspect recurring log features associated with this prediction.",
        ]

    if category == "Resource/Capacity Stress":
        return [
            "Inspect resource allocation and capacity utilization.",
            "Check for hardware saturation or overloaded resource pools.",
            "Verify whether this resource type is repeatedly linked to severe faults.",
        ]

    if category == "Alarm Burst / Incident Cascade":
        return [
            "Inspect correlated alarms occurring in the same time window.",
            "Check whether one incident triggered multiple downstream events.",
            "Review event-type sequences for cascading failures.",
        ]

    if category == "Historical Severity Pattern":
        return [
            "Inspect whether the severity profile matches known severe cases.",
            "Compare with historical incidents having the same severity type.",
            "Review incident history for recurring escalation patterns.",
        ]

    return [
        "Review top contributing features manually.",
        "Inspect related logs, events, and resource signals.",
        "Cross-check the case with previous known fault patterns.",
    ]


# 4. Load Model and Data

# Load final trained model
final_model = joblib.load(MODEL_PATH)

# Load train, validation, and test splits
X_train = pd.read_csv(SPLITS_PATH / "X_train.csv")
X_valid = pd.read_csv(SPLITS_PATH / "X_valid.csv")
X_test = pd.read_csv(SPLITS_PATH / "X_test.csv")

y_train = pd.read_csv(SPLITS_PATH / "y_train.csv")["fault_severity"]
y_valid = pd.read_csv(SPLITS_PATH / "y_valid.csv")["fault_severity"]
y_test = pd.read_csv(SPLITS_PATH / "y_test.csv")["fault_severity"]

# Load metadata and final test predictions
meta_test = pd.read_csv(SPLITS_PATH / "meta_test.csv")
pred_df = pd.read_csv(PREDS_PATH)

# Recreate final train data used in notebook 07
X_final_train = pd.concat([X_train, X_valid], axis=0, ignore_index=True)
y_final_train = pd.concat([y_train, y_valid], axis=0, ignore_index=True)

feature_names = X_test.columns.tolist()
class_labels = sorted(y_final_train.unique().tolist())

print("\nLoaded objects:")
print("X_final_train:", X_final_train.shape)
print("X_test       :", X_test.shape)
print("y_test       :", y_test.shape)
print("meta_test    :", meta_test.shape)
print("pred_df      :", pred_df.shape)
print("n_features   :", len(feature_names))
print("classes      :", class_labels)


# 5. Build SHAP Explainer

print("\nBUILDING SHAP EXPLAINER")

# SHAP explains how each feature contributes to predictions
explainer = shap.TreeExplainer(final_model)
shap_values_raw = explainer.shap_values(X_test)

# Standardize SHAP output format
shap_by_class = normalize_shap_output(shap_values_raw, class_labels)

for cls in class_labels:
    print(f"SHAP matrix for class {cls}: {shap_by_class[cls].shape}")


# 6. Global Feature Importance

print("\nGLOBAL FEATURE IMPORTANCE")

# Compute overall mean absolute SHAP importance across all classes
overall_mean_abs = np.mean(
    np.stack(
        [np.abs(shap_by_class[cls]).mean(axis=0) for cls in class_labels],
        axis=0,
    ),
    axis=0,
)

overall_global_df = pd.DataFrame(
    {
        "feature": feature_names,
        "mean_abs_shap_overall": overall_mean_abs,
    }
)

overall_global_df["feature_group"] = overall_global_df["feature"].apply(detect_feature_group)
overall_global_df = overall_global_df.sort_values("mean_abs_shap_overall", ascending=False).reset_index(drop=True)

print(overall_global_df.head(20))

# Save overall global feature importance
save_csv(overall_global_df, REPORTS_PATH / "global_feature_importance_overall.csv", index=False)

# Group feature importance by feature group
group_summary = (
    overall_global_df.groupby("feature_group")["mean_abs_shap_overall"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("\nFeature-group importance:")
print(group_summary)

save_csv(group_summary, REPORTS_PATH / "global_feature_group_importance.csv", index=False)


# 7. Local Explanations for Selected Cases

print("\nLOCAL EXPLANATIONS")

# Confidence = highest predicted class probability
pred_df["pred_confidence"] = pred_df[[f"prob_class_{c}" for c in class_labels]].max(axis=1)

# Mark whether prediction is correct
pred_df["is_correct"] = pred_df["y_true"] == pred_df["y_pred"]

# Choose a few easy example cases for viva:
# - 3 highest-confidence correct predictions
# - 3 highest-confidence wrong predictions
correct_cases = pred_df[pred_df["is_correct"]].sort_values("pred_confidence", ascending=False).head(3)
wrong_cases = pred_df[~pred_df["is_correct"]].sort_values("pred_confidence", ascending=False).head(3)

selected_cases = pd.concat([correct_cases, wrong_cases], axis=0).reset_index(drop=True)

# Save selected cases
save_csv(selected_cases, REPORTS_PATH / "selected_explanation_cases.csv", index=False)

print(selected_cases[["id", "location", "y_true", "y_pred", "pred_confidence", "is_correct"]])

local_explanations = []

# Build local explanation summary for each selected case
for _, row in selected_cases.iterrows():
    case_id = int(row["id"])
    case_idx = pred_df.index[pred_df["id"] == case_id][0]
    pred_class = int(row["y_pred"])

    # Get top contributing features for the predicted class
    top_df = get_top_features_for_row(
        shap_matrix=shap_by_class[pred_class],
        feature_names=feature_names,
        row_index=case_idx,
        top_n=10,
    )

    category = infer_fault_category(top_df)
    isolation_summary = build_isolation_summary(top_df, pred_class)
    recommended_checks = build_recommended_checks(category)

    top_df["feature_group"] = top_df["feature"].apply(detect_feature_group)

    # Save one row per recommendation so the output is structured
    for rank, rec in enumerate(recommended_checks, start=1):
        local_explanations.append(
            {
                "id": case_id,
                "location": row["location"],
                "y_true": int(row["y_true"]),
                "y_pred": pred_class,
                "pred_confidence": float(row["pred_confidence"]),
                "is_correct": bool(row["is_correct"]),
                "fault_category": category,
                "isolation_summary": isolation_summary,
                "top_features": ", ".join(top_df["feature"].head(5).tolist()),
                "recommended_check_rank": rank,
                "recommended_check": rec,
            }
        )

local_explanations_df = pd.DataFrame(local_explanations)

# Save local explanation summary
save_csv(local_explanations_df, REPORTS_PATH / "local_explanations_summary.csv", index=False)


# 8. Full Test-Set Isolation Outputs

print("\nFULL TEST-SET ISOLATION OUTPUTS")

full_isolation_rows = []

# Build isolation output for every test case
for i in range(len(X_test)):
    pred_class = int(pred_df.loc[i, "y_pred"])

    top_df = get_top_features_for_row(
        shap_matrix=shap_by_class[pred_class],
        feature_names=feature_names,
        row_index=i,
        top_n=10,
    )

    category = infer_fault_category(top_df)
    summary = build_isolation_summary(top_df, pred_class)
    checks = build_recommended_checks(category)

    full_isolation_rows.append(
        {
            "id": int(pred_df.loc[i, "id"]),
            "location": pred_df.loc[i, "location"],
            "y_true": int(pred_df.loc[i, "y_true"]),
            "y_pred": pred_class,
            "prob_class_0": float(pred_df.loc[i, "prob_class_0"]),
            "prob_class_1": float(pred_df.loc[i, "prob_class_1"]),
            "prob_class_2": float(pred_df.loc[i, "prob_class_2"]),
            "pred_confidence": float(pred_df.loc[i, "pred_confidence"]),
            "fault_category": category,
            "isolation_summary": summary,
            "top_features": ", ".join(top_df["feature"].head(5).tolist()),
            "recommended_checks": " | ".join(checks),
        }
    )

full_isolation_df = pd.DataFrame(full_isolation_rows)
print(full_isolation_df.head())

# Save full isolation output table
save_csv(full_isolation_df, ARTIFACTS_PATH / "full_test_isolation_outputs.csv", index=False)


# 9. Fault Category Distribution

print("\nCATEGORY DISTRIBUTION")

# Count how often each inferred category appears in the full test set
category_dist = (
    full_isolation_df["fault_category"]
    .value_counts()
    .rename_axis("fault_category")
    .reset_index(name="count")
)

category_dist["proportion"] = (category_dist["count"] / len(full_isolation_df)).round(4)

print(category_dist)

# Save category distribution
save_csv(category_dist, REPORTS_PATH / "fault_category_distribution.csv", index=False)


# 10. Compact Explainability Summary

print("\nEXPLAINABILITY SUMMARY")

# Small summary table 
summary_df = pd.DataFrame(
    {
        "item": [
            "n_test_cases_explained",
            "n_features",
            "n_classes",
            "top_global_feature",
            "top_global_feature_group",
            "most_common_fault_category",
        ],
        "value": [
            len(X_test),
            len(feature_names),
            len(class_labels),
            overall_global_df.iloc[0]["feature"],
            overall_global_df.iloc[0]["feature_group"],
            category_dist.iloc[0]["fault_category"],
        ],
    }
)

print(summary_df)

save_csv(summary_df, REPORTS_PATH / "explainability_notebook_summary.csv", index=False)

print("\nEXPLAINABILITY AND FAULT ISOLATION COMPLETE")
print("Reports saved to :", REPORTS_PATH)
print("Artifacts saved to:", ARTIFACTS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
SHAP   : 0.50.0
PROJECT_ROOT : /Users/hasheenadilmidesilva/Desktop/NetGuard
SPLITS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/splits
MODEL_PATH   : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/final_model/lightgbm_final_model.joblib
PREDS_PATH   : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/final_predictions/final_test_predictions.csv
REPORTS_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/explainability
ARTIFACTS    : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/explainability

Loaded objects:
X_final_train: (5904, 860)
X_test       : (1477, 860)
y_test       : (1477,)
meta_test    : (1477, 2)
pred_df      : (1477, 7)
n_features   : 860
classes      : [0, 1, 2]

BUILDING SHAP EXPLAINER
SHAP matrix for class 0: (1477, 860)
SHAP matrix for class 1: (1477, 860)
SHAP matrix for class 2: (1477, 860)

GLOBAL FEATURE IMPORTANCE
                 feature  mean_abs_shap_overall         featur